In [ ]:
!pip install kagglehub torch torchvision --quiet

import kagglehub
import os

# Authenticate kagglehub (Enter your API key or token when prompted)
kagglehub.login()

# Download dataset directly from Kaggle
print("Downloading dataset from Kaggle...")
dataset_path = kagglehub.dataset_download("preetpalsingh25/alzheimers-dataset-4-class-of-images")
print("\nDataset downloaded to Colab path:", dataset_path)

# Check the subdirectories inside the dataset path
print("Folder contents:", os.listdir(dataset_path))

100%|██████████| 56.4M/56.4M [00:00<00:00, 64.1MB/s]

Extracting files...



Dataset downloaded to Colab path: /root/.cache/kagglehub/datasets/preetpalsingh25/alzheimers-dataset-4-class-of-images/versions/1
Folder contents: ['Alzheimer_s Dataset']
Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader

# 1. Define folder paths (handles nested dataset directories automatically)
train_dir = os.path.join(dataset_path, "train")
test_dir  = os.path.join(dataset_path, "test")

# Fallback check if directories are nested inside an inner folder
if not os.path.exists(train_dir):
    subfolders = [f for f in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, f))]
    if subfolders:
        inner_path = os.path.join(dataset_path, subfolders[0])
        train_dir = os.path.join(inner_path, "train")
        test_dir  = os.path.join(inner_path, "test")

print("Using Train Directory:", train_dir)
print("Using Test Directory:", test_dir)

# 2. Image Preprocessing Transforms
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 3. Load Image Datasets
train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transforms)
test_dataset  = datasets.ImageFolder(root=test_dir, transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

class_names = train_dataset.classes
print(f"\nDetected Classes ({len(class_names)}): {class_names}")

# 4. Initialize ResNet-18 Model with GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(class_names))
model = model.to(device)

# 5. Loss & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

# 6. Training Loop (5 Epochs)
epochs = 5
print("\nStarting Training Loop...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {running_loss/len(train_loader):.4f} - Accuracy: {100 * correct / total:.2f}%")

# 7. Save Weights File
model_filename = 'alzheimer_mri_resnet18.pth'
torch.save(model.state_dict(), model_filename)
print(f"\nTraining Complete! Model saved as '{model_filename}' in Colab.")

Using Train Directory: /root/.cache/kagglehub/datasets/preetpalsingh25/alzheimers-dataset-4-class-of-images/versions/1/Alzheimer_s Dataset/train
Using Test Directory: /root/.cache/kagglehub/datasets/preetpalsingh25/alzheimers-dataset-4-class-of-images/versions/1/Alzheimer_s Dataset/test

Detected Classes (4): ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
Training on device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 132MB/s]



Starting Training Loop...
Epoch [1/5] - Loss: 0.7059 - Accuracy: 70.16%
Epoch [2/5] - Loss: 0.2991 - Accuracy: 88.42%
Epoch [3/5] - Loss: 0.1492 - Accuracy: 94.50%
Epoch [4/5] - Loss: 0.0753 - Accuracy: 97.22%
Epoch [5/5] - Loss: 0.0583 - Accuracy: 98.14%

Training Complete! Model saved as 'alzheimer_mri_resnet18.pth' in Colab.


In [ ]:
# 1. Evaluate on Test Set
model.eval()
test_correct = 0
test_total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

print(f"Final Test Set Accuracy: {100 * test_correct / test_total:.2f}%")

# 2. Download weights file to your PC
from google.colab import files
files.download('alzheimer_mri_resnet18.pth')

Final Test Set Accuracy: 98.70%


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>